In [19]:
import os
import numpy as np
from tqdm import tqdm
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

INPUT_DIR = "./data"
OUTPUT_DIR = "./dataset_resized"
DENOISED_DIR = "./dataset_denoised"
TARGET_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DENOISED_DIR, exist_ok=True)

def resize_images(input_dir, output_dir, size=256):
    for root, _, files in os.walk(input_dir):
        rel = os.path.relpath(root, input_dir)
        save_dir = os.path.join(output_dir, rel)
        os.makedirs(save_dir, exist_ok=True)

        for f in tqdm(files, desc=f"Resizing {rel}"):
            if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            try:
                img = Image.open(os.path.join(root, f)).convert("RGB")
                img = img.resize((size, size), Image.BICUBIC)
                img.save(os.path.join(save_dir, f))
            
            except Exception as e:
                print(f"Error processing {f}: {e}")

resize_images(INPUT_DIR, OUTPUT_DIR, TARGET_SIZE)

Resizing CORROSION:  47%|████▋     | 935/1980 [00:00<00:00, 9342.15it/s]

Error processing ._000001.jpg: cannot identify image file './data/CORROSION/._000001.jpg'
Error processing ._000004.jpg: cannot identify image file './data/CORROSION/._000004.jpg'
Error processing ._000006.jpg: cannot identify image file './data/CORROSION/._000006.jpg'
Error processing ._000007.jpg: cannot identify image file './data/CORROSION/._000007.jpg'
Error processing ._000008.jpg: cannot identify image file './data/CORROSION/._000008.jpg'
Error processing ._000009.jpg: cannot identify image file './data/CORROSION/._000009.jpg'
Error processing ._000010.jpg: cannot identify image file './data/CORROSION/._000010.jpg'
Error processing ._000012.jpg: cannot identify image file './data/CORROSION/._000012.jpg'
Error processing ._000013.jpg: cannot identify image file './data/CORROSION/._000013.jpg'
Error processing ._000014.jpg: cannot identify image file './data/CORROSION/._000014.jpg'
Error processing ._000015.jpg: cannot identify image file './data/CORROSION/._000015.jpg'
Error proc

Resizing NOCORROSION:  50%|████▉     | 820/1644 [00:00<00:00, 8180.23it/s]

Error processing ._01401e0891.jpg: cannot identify image file './data/NOCORROSION/._01401e0891.jpg'
Error processing ._01a5aee1ab.jpg: cannot identify image file './data/NOCORROSION/._01a5aee1ab.jpg'
Error processing ._037f0f3b6a.jpg: cannot identify image file './data/NOCORROSION/._037f0f3b6a.jpg'
Error processing ._039219204d.jpg: cannot identify image file './data/NOCORROSION/._039219204d.jpg'
Error processing ._03c66b67be.jpg: cannot identify image file './data/NOCORROSION/._03c66b67be.jpg'
Error processing ._05977f2945.jpg: cannot identify image file './data/NOCORROSION/._05977f2945.jpg'
Error processing ._05b801c3ce.jpg: cannot identify image file './data/NOCORROSION/._05b801c3ce.jpg'
Error processing ._05bc6d28ff.jpg: cannot identify image file './data/NOCORROSION/._05bc6d28ff.jpg'
Error processing ._05ec049237.jpg: cannot identify image file './data/NOCORROSION/._05ec049237.jpg'
Error processing ._062a79b3ab.jpg: cannot identify image file './data/NOCORROSION/._062a79b3ab.jpg'


Resizing NOCORROSION: 100%|██████████| 1644/1644 [00:07<00:00, 208.98it/s]


In [20]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [21]:
def load_images_from_folder(folder, size=256, limit=None):
    images = []
    for root, _, files in os.walk(folder):
        for f in files:
            if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img = Image.open(os.path.join(root, f)).convert("RGB")
            img = img.resize((size, size))
            images.append(np.array(img))
            if limit and len(images) >= limit:
                break
    return np.array(images, dtype=np.float32) / 255.0

print("📥 Loading dataset...")
X = load_images_from_folder(OUTPUT_DIR, TARGET_SIZE)
print(f"✅ Loaded {len(X)} images at {TARGET_SIZE}×{TARGET_SIZE}")

📥 Loading dataset...
✅ Loaded 1819 images at 256×256


In [22]:
# ======================================
# 🧠 4. Add synthetic noise for training
# ======================================

noise_factor = 0.15
X_noisy = X + noise_factor * np.random.randn(*X.shape)
X_noisy = np.clip(X_noisy, 0., 1.)

In [23]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

In [24]:
# ======================================
# 🏗️ 5. Build Denoising Autoencoder (DAE)
# ======================================

def build_denoising_autoencoder(img_shape=(256, 256, 3)):
    input_img = layers.Input(shape=img_shape)

    # Encoder
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(input_img)
    x = layers.MaxPooling2D((2,2), padding='same')(x)
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
    encoded = layers.MaxPooling2D((2,2), padding='same')(x)

    # Decoder
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(encoded)
    x = layers.UpSampling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2,2))(x)
    decoded = layers.Conv2D(3, (3,3), activation='sigmoid', padding='same')(x)

    model = models.Model(input_img, decoded)
    model.compile(optimizer=optimizers.Adam(1e-4), loss='mse')
    return model

In [25]:
autoencoder = build_denoising_autoencoder((TARGET_SIZE, TARGET_SIZE, 3))
autoencoder.summary()

2025-11-03 13:43:43.155452: W tensorflow/compiler/mlir/tools/kernel_gen/tf_gpu_runtime_wrappers.cc:40] 'cuModuleLoadData(&module, data)' failed with 'CUDA_ERROR_UNSUPPORTED_PTX_VERSION'

2025-11-03 13:43:43.155494: W tensorflow/compiler/mlir/tools/kernel_gen/tf_gpu_runtime_wrappers.cc:40] 'cuModuleGetFunction(&function, module, kernel_name)' failed with 'CUDA_ERROR_INVALID_HANDLE'

2025-11-03 13:43:43.155507: W tensorflow/core/framework/op_kernel.cc:1842] INTERNAL: 'cuLaunchKernel(function, gridX, gridY, gridZ, blockX, blockY, blockZ, 0, reinterpret_cast<CUstream>(stream), params, nullptr)' failed with 'CUDA_ERROR_INVALID_HANDLE'


InternalError: {{function_node __wrapped__FloorMod_device_/job:localhost/replica:0/task:0/device:GPU:0}} 'cuLaunchKernel(function, gridX, gridY, gridZ, blockX, blockY, blockZ, 0, reinterpret_cast<CUstream>(stream), params, nullptr)' failed with 'CUDA_ERROR_INVALID_HANDLE' [Op:FloorMod] name: 

In [9]:
python -c "import tensorflow as tf; print(tf.__version__)"

SyntaxError: invalid syntax (71107497.py, line 1)